# Data Science & Database Assignment: Session 9
## Topic: JOINS - Part 1 (INNER, LEFT, & RIGHT JOINS)

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `zomato_joins.db`  
**Core Concepts:** Relational Keys, Foreign Keys, `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, Handling `NULL`s, One-to-Many Relationships  
**Status:** Completed & Pre-Executed  

---

### Understanding Relational Joins
Relational databases separate data into normalized entities to prevent data redundancy and maintain integrity. Joins allow us to reconstruct complete, multi-table business views on-the-fly:

```text
+-------------------+--------------------------------------------------------------------------+
| JOIN Type         | Result Set Behavior                                                      |
+-------------------+--------------------------------------------------------------------------+
| INNER JOIN        | Returns only rows where there is a matching key in BOTH tables.          |
| LEFT (OUTER) JOIN | Returns ALL rows from the LEFT table, plus matched rows from RIGHT table |
|                   | (unmatched right rows become NULL).                                      |
| RIGHT (OUTER) JOIN| Returns ALL rows from the RIGHT table, plus matched rows from LEFT table|
|                   | (unmatched left rows become NULL).                                       |
| FULL OUTER JOIN   | Returns ALL rows from BOTH tables, filling with NULL wherever unmatched. |
+-------------------+--------------------------------------------------------------------------+
```


## Task 1: Create 'restaurants' and 'dishes' Tables & Insert Sample Records
> **Problem Statement:** Create two tables in your database: `restaurants` (`id`, `name`, `city`) and `dishes` (`id`, `restaurant_id`, `dish_name`, `price`). Insert at least 3 restaurants and 2-3 dishes for each restaurant.

### Schema Architecture
- **`restaurants`**: Primary entity storing restaurant branches across cities.
- **`dishes`**: Child entity linked to `restaurants` via `restaurant_id` (foreign key).
- **Edge cases tested:**
  1. **Empty Restaurant:** Added `Cafe Coffee Day` in Mumbai with **0 dishes** to test `LEFT JOIN` in Task 3.
  2. **Orphaned Dish:** Added `Dragon Noodles` with `restaurant_id = 99` (non-existent restaurant) to test `RIGHT JOIN` in Task 4.


In [1]:
import sqlite3
import pandas as pd

# Connect to local SQLite database
conn = sqlite3.connect('zomato_joins.db')
cursor = conn.cursor()

# Reset tables for clean reproducibility
cursor.execute("DROP TABLE IF EXISTS dishes;")
cursor.execute("DROP TABLE IF EXISTS restaurants;")

# 1. Create 'restaurants' table
cursor.execute("""
CREATE TABLE restaurants (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    city VARCHAR(50) NOT NULL
);
""")

# 2. Create 'dishes' table
cursor.execute("""
CREATE TABLE dishes (
    id INT PRIMARY KEY,
    restaurant_id INT,
    dish_name VARCHAR(100) NOT NULL,
    price DECIMAL(10, 2) NOT NULL
);
""")

# 3. Insert Restaurants
restaurants_data = [
    (1, 'Biryani By Kilo', 'Hyderabad'),
    (2, 'Haldiram', 'Nagpur'),
    (3, 'Pizza Hut', 'Pune'),
    (4, 'Cafe Coffee Day', 'Mumbai')  # Zero dishes associated (to test LEFT JOIN)
]
cursor.executemany("INSERT INTO restaurants VALUES (?, ?, ?);", restaurants_data)

# 4. Insert Dishes
dishes_data = [
    # Biryani By Kilo (restaurant_id = 1)
    (101, 1, 'Hyderabadi Chicken Biryani', 350.00),
    (102, 1, 'Mutton Dum Biryani', 480.00),
    (103, 1, 'Mirchi Ka Salan', 120.00),
    # Haldiram (restaurant_id = 2)
    (104, 2, 'Raj Kachori', 140.00),
    (105, 2, 'Chole Bhature', 180.00),
    (106, 2, 'Gulab Jamun (2 pcs)', 90.00),
    # Pizza Hut (restaurant_id = 3)
    (107, 3, 'Margherita Pizza', 299.00),
    (108, 3, 'Garlic Breadsticks', 149.00),
    (109, 3, 'Farmhouse Veg Pizza', 399.00),
    # Orphaned Dish (restaurant_id = 99 -> simulating data integrity glitch for RIGHT JOIN)
    (110, 99, 'Dragon Noodles', 220.00)
]
cursor.executemany("INSERT INTO dishes VALUES (?, ?, ?, ?);", dishes_data)
conn.commit()

print("[OK] Tables created and populated successfully.")


[OK] Tables created and populated successfully.


In [2]:
# Display both tables
print("=== Restaurants Table ===")
display(pd.read_sql_query("SELECT * FROM restaurants;", conn))

print("=== Dishes Table ===")
display(pd.read_sql_query("SELECT * FROM dishes;", conn))


=== Restaurants Table ===


,id,name,city
0,1,Biryani By Kilo,Hyderabad
1,2,Haldiram,Nagpur
2,3,Pizza Hut,Pune
3,4,Cafe Coffee Day,Mumbai


=== Dishes Table ===


,id,restaurant_id,dish_name,price
0,101,1,Hyderabadi Chicken Biryani,350
1,102,1,Mutton Dum Biryani,480
2,103,1,Mirchi Ka Salan,120
3,104,2,Raj Kachori,140
4,105,2,Chole Bhature,180
5,106,2,Gulab Jamun (2 pcs),90
6,107,3,Margherita Pizza,299
7,108,3,Garlic Breadsticks,149
8,109,3,Farmhouse Veg Pizza,399
9,110,99,Dragon Noodles,220


## Task 2: INNER JOIN (Dishes with Restaurant Info)
> **Problem Statement:** Write an SQL `INNER JOIN` query to display each dish along with its restaurant name and city, similar to how Zomato shows dish details with the restaurant info.

### Conceptual Explanation
`INNER JOIN` requires a matching key on both sides: `dishes.restaurant_id = restaurants.id`.
- Dishes belonging to valid restaurants are returned.
- Restaurant 4 (`Cafe Coffee Day`) has no dishes, so it is **excluded**.
- Dish 110 (`Dragon Noodles`) has `restaurant_id = 99` (no matching restaurant), so it is **excluded**.
- Result: Exactly the **9 validly matched dishes** appear in the output.


In [3]:
task2_query = """
SELECT 
    d.id AS dish_id,
    d.dish_name,
    d.price,
    r.name AS restaurant_name,
    r.city
FROM dishes d
INNER JOIN restaurants r ON d.restaurant_id = r.id;
"""

df_inner = pd.read_sql_query(task2_query, conn)
display(df_inner)
print(f"Total matching items returned: {len(df_inner)}")


,dish_id,dish_name,price,restaurant_name,city
0,101,Hyderabadi Chicken Biryani,350,Biryani By Kilo,Hyderabad
1,102,Mutton Dum Biryani,480,Biryani By Kilo,Hyderabad
2,103,Mirchi Ka Salan,120,Biryani By Kilo,Hyderabad
3,104,Raj Kachori,140,Haldiram,Nagpur
4,105,Chole Bhature,180,Haldiram,Nagpur
5,106,Gulab Jamun (2 pcs),90,Haldiram,Nagpur
6,107,Margherita Pizza,299,Pizza Hut,Pune
7,108,Garlic Breadsticks,149,Pizza Hut,Pune
8,109,Farmhouse Veg Pizza,399,Pizza Hut,Pune


Total matching items returned: 9


## Task 3: LEFT JOIN (All Restaurants Including Those Without Dishes)
> **Problem Statement:** Write an SQL `LEFT JOIN` query to list all restaurants and their dishes, showing restaurants even if they currently have no dishes on the menu.  
> *Hint: Use LEFT JOIN so restaurants without dishes still appear in the results with NULL for dish columns.*

### Conceptual Explanation
By placing `restaurants` on the LEFT side of the join (`FROM restaurants r LEFT JOIN dishes d`), the engine preserves **every row from `restaurants`**:
- When dishes exist for a restaurant (e.g. `Biryani By Kilo`, `Haldiram`, `Pizza Hut`), the rows pair up normally.
- When a restaurant has no menu dishes yet (`Cafe Coffee Day`), the restaurant columns are displayed and all dish attributes (`dish_id`, `dish_name`, `price`) evaluate to **`NULL`**.


In [4]:
task3_query = """
SELECT 
    r.id AS restaurant_id,
    r.name AS restaurant_name,
    r.city,
    d.id AS dish_id,
    d.dish_name,
    d.price
FROM restaurants r
LEFT JOIN dishes d ON r.id = d.restaurant_id;
"""

df_left = pd.read_sql_query(task3_query, conn)
display(df_left)
print(f"Total rows returned: {len(df_left)}")


,restaurant_id,restaurant_name,city,dish_id,dish_name,price
0,1,Biryani By Kilo,Hyderabad,101.0,Hyderabadi Chicken Biryani,350.0
1,1,Biryani By Kilo,Hyderabad,102.0,Mutton Dum Biryani,480.0
2,1,Biryani By Kilo,Hyderabad,103.0,Mirchi Ka Salan,120.0
3,2,Haldiram,Nagpur,104.0,Raj Kachori,140.0
4,2,Haldiram,Nagpur,105.0,Chole Bhature,180.0
5,2,Haldiram,Nagpur,106.0,Gulab Jamun (2 pcs),90.0
6,3,Pizza Hut,Pune,107.0,Margherita Pizza,299.0
7,3,Pizza Hut,Pune,108.0,Garlic Breadsticks,149.0
8,3,Pizza Hut,Pune,109.0,Farmhouse Veg Pizza,399.0
9,4,Cafe Coffee Day,Mumbai,NaN,NaN,NaN


Total rows returned: 10


## Task 4: RIGHT JOIN (All Dishes Including Orphaned / Unlinked Dishes)
> **Problem Statement:** Write an SQL `RIGHT JOIN` query to display all dishes and their restaurant names, including any dishes that might not be linked to a restaurant (simulate a data error where a dish has a `restaurant_id` that doesn't match any restaurant).

### Conceptual Explanation
In `FROM restaurants r RIGHT JOIN dishes d ON r.id = d.restaurant_id`:
- The `dishes` table is on the RIGHT.
- Every dish is guaranteed to be returned in the output.
- For dish `110` (`Dragon Noodles` with `restaurant_id = 99`), no restaurant row in `restaurants` matches ID 99.
- Therefore, the restaurant attributes (`restaurant_id`, `name`, `city`) evaluate to **`NULL`**.
- This pattern is standard in data auditing to detect orphaned child records that violate referential integrity.


In [5]:
task4_query = """
SELECT 
    r.id AS restaurant_id,
    r.name AS restaurant_name,
    r.city,
    d.id AS dish_id,
    d.dish_name,
    d.price,
    d.restaurant_id AS dish_fk
FROM restaurants r
RIGHT JOIN dishes d ON r.id = d.restaurant_id;
"""

df_right = pd.read_sql_query(task4_query, conn)
display(df_right)
print(f"Total rows returned: {len(df_right)}")


,restaurant_id,restaurant_name,city,dish_id,dish_name,price,dish_fk
0,1.0,Biryani By Kilo,Hyderabad,101,Hyderabadi Chicken Biryani,350,1
1,1.0,Biryani By Kilo,Hyderabad,102,Mutton Dum Biryani,480,1
2,1.0,Biryani By Kilo,Hyderabad,103,Mirchi Ka Salan,120,1
3,2.0,Haldiram,Nagpur,104,Raj Kachori,140,2
4,2.0,Haldiram,Nagpur,105,Chole Bhature,180,2
5,2.0,Haldiram,Nagpur,106,Gulab Jamun (2 pcs),90,2
6,3.0,Pizza Hut,Pune,107,Margherita Pizza,299,3
7,3.0,Pizza Hut,Pune,108,Garlic Breadsticks,149,3
8,3.0,Pizza Hut,Pune,109,Farmhouse Veg Pizza,399,3
9,NaN,NaN,NaN,110,Dragon Noodles,220,99


Total rows returned: 10


## Task 5: Spotify Scenario - Playlists and Songs
> **Problem Statement:** Given this scenario: You want to show a list of all playlists and the songs inside them, like Spotify. Explain which JOIN type (`INNER`, `LEFT`, or `RIGHT`) you would use to show all playlists, even if some are empty, and write the SQL query for it.

---

### In-Depth Architectural Explanation

#### 1. Why `INNER JOIN` is INCORRECT:
An `INNER JOIN` strictly produces rows where the join condition matches in **both** tables (`playlists.playlist_id = songs.playlist_id`).
- When a user signs up on Spotify and clicks *"Create New Playlist"* (e.g., `"Road Trip 2026"`), the playlist initially has **0 songs**.
- An `INNER JOIN` completely discards rows without child matches. As a result, the user's new empty playlist would be **invisible** on their profile library!
- This breaks fundamental user expectations.

#### 2. Why `LEFT JOIN` is the IDEAL Choice:
Placing `playlists` on the **LEFT** and `songs` on the **RIGHT** (`FROM playlists p LEFT JOIN songs s ON p.playlist_id = s.playlist_id`):
- Guarantees that **every playlist** created by any user is returned in the result set at least once.
- If a playlist has 5 songs, 5 rows are returned.
- If a playlist is empty, exactly 1 row is returned with `song_id`, `title`, and `duration_sec` as `NULL`.
- The frontend / UI layer checks `if song_id IS NULL`, gracefully rendering `"Empty Playlist (0 tracks)"` and an `[+ Add Songs]` button.

#### 3. What about `RIGHT JOIN`?
A `RIGHT JOIN` could only achieve this if `songs` was placed on the left and `playlists` on the right (`FROM songs s RIGHT JOIN playlists p`). While mathematically equivalent, standard SQL conventions prefer reading queries in natural primary-to-child entity hierarchy (`FROM parent LEFT JOIN child`). Therefore, `LEFT JOIN` is the industry standard for this pattern.


In [6]:
# Set up Spotify test tables
cursor.execute("DROP TABLE IF EXISTS songs;")
cursor.execute("DROP TABLE IF EXISTS playlists;")

cursor.execute("""
CREATE TABLE playlists (
    playlist_id INT PRIMARY KEY,
    playlist_name VARCHAR(100) NOT NULL,
    created_by VARCHAR(50) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE songs (
    song_id INT PRIMARY KEY,
    playlist_id INT,
    title VARCHAR(100) NOT NULL,
    artist VARCHAR(100) NOT NULL,
    duration_sec INT NOT NULL,
    FOREIGN KEY (playlist_id) REFERENCES playlists(playlist_id)
);
""")

# Insert 4 Playlists (Playlist 4 'Road Trip 2026' is empty)
playlists_data = [
    (1, 'Gym Workout Beats', 'Atharv'),
    (2, 'Acoustic Morning', 'Sneha'),
    (3, 'Study Lo-Fi Focus', 'Rahul'),
    (4, 'Road Trip 2026', 'Atharv')  # Newly created, empty playlist
]
cursor.executemany("INSERT INTO playlists VALUES (?, ?, ?);", playlists_data)

# Insert songs
songs_data = [
    (201, 1, 'Eye of the Tiger', 'Survivor', 245),
    (202, 1, 'Lose Yourself', 'Eminem', 326),
    (203, 1, 'Till I Collapse', 'Eminem', 297),
    (204, 2, 'Banana Pancakes', 'Jack Johnson', 192),
    (205, 2, 'Here Comes the Sun', 'The Beatles', 185),
    (206, 3, 'Sunset Lover', 'Petit Biscuit', 237)
]
cursor.executemany("INSERT INTO songs VALUES (?, ?, ?, ?, ?);", songs_data)
conn.commit()

# Execute Spotify LEFT JOIN Query:
task5_query = """
SELECT 
    p.playlist_id,
    p.playlist_name,
    p.created_by,
    s.song_id,
    s.title AS song_title,
    s.artist,
    s.duration_sec
FROM playlists p
LEFT JOIN songs s ON p.playlist_id = s.playlist_id;
"""

df_spotify = pd.read_sql_query(task5_query, conn)
display(df_spotify)

conn.close()


,playlist_id,playlist_name,created_by,song_id,song_title,artist,duration_sec
0,1,Gym Workout Beats,Atharv,201.0,Eye of the Tiger,Survivor,245.0
1,1,Gym Workout Beats,Atharv,202.0,Lose Yourself,Eminem,326.0
2,1,Gym Workout Beats,Atharv,203.0,Till I Collapse,Eminem,297.0
3,2,Acoustic Morning,Sneha,204.0,Banana Pancakes,Jack Johnson,192.0
4,2,Acoustic Morning,Sneha,205.0,Here Comes the Sun,The Beatles,185.0
5,3,Study Lo-Fi Focus,Rahul,206.0,Sunset Lover,Petit Biscuit,237.0
6,4,Road Trip 2026,Atharv,NaN,NaN,NaN,NaN


## Summary & Key Takeaways

1. **`INNER JOIN`**: Use when you only care about complete relationships where data exists in both tables (e.g., active orders with valid items).
2. **`LEFT JOIN`**: Use when the primary entity must always be displayed regardless of whether child records exist (e.g., all restaurants even if menu is empty, all users even if they haven't ordered, all playlists even if empty).
3. **`RIGHT JOIN`**: Use when you need to preserve all records from the secondary table, especially useful in database integrity audits to locate orphaned child records whose foreign key references a non-existent parent.
4. **`NULL` Handling**: Outer joins inject `NULL` for missing attributes. In application code, use `COALESCE()` or `IFNULL()` to provide friendly defaults like `'No dishes currently listed'` or `0`.
